# 10 - Post-training quantization (PTQ) vs quantization-aware training (QAT)
Reviewer 4 comment 2 / Reviewer 1 comment 1 of the response letter. Same 11-16-8-1 network, same data split, same scaling, same hls4ml conversion for PTQ and QAT; all variants are evaluated with the **hls4ml bit-accurate simulation** (round-to-nearest, saturation).
QAT = the trained float network fine-tuned for 30 epochs, learning rate 0.0005, fixed-point rounding + saturation in the forward pass, straight-through gradient.

**Values quoted in the letter (what this notebook should reproduce, +-a few tenths of a point):**

| set | float | PTQ 8 / QAT 8 | PTQ 6 / QAT 6 | 4 bit |
|---|---|---|---|---|
| original windows, 348-window test set | 97.99 | 97.99 / 98.28 | 96.55 / 97.13 | 34.20 / 34.20 |
| original windows, 5-fold CV over subjects | 96.60 +- 2.20 | 96.37 / 96.79 | 95.55 / 95.98 | 44.21 / 44.21 |
| augmented windows, 5-fold CV over subjects | 95.64 +- 3.00 | 95.74 / 95.83 | 94.72 / 95.29 | 44.42 / 44.42 |
| augmented, 1,394-window test, paper network | 98.06 | 98.35 | 97.63 | (12 bit 98.13, 16 bit 98.06) |
| augmented, 1,394-window test, a re-trained float network | 98.28 | 97.27 / 97.70 | 96.05 / 96.70 | |

Runtime: roughly 1-2 hours (many hls4ml compilations). Needs a Linux runtime with g++ (Colab is fine).

**How to run (Google Colab, the same place the other notebooks ran).** Folder `MyDrive/ppg_afib_project/` must contain the two PERform zips (`mimic_perform_af_csv.zip`, `mimic_perform_non_af_csv.zip`, searched recursively). Run the cells top to bottom with *Runtime -> Run all*. **Do not edit the setup cell and do not retrain or replace `ppg_afib_model_augmented.h5`.** When it finishes, *File -> Download -> Download .ipynb* (with the outputs) and send the executed notebook back, together with the CSV that is written to `ppg_afib_project/augmented_window_results/`.
If a number differs from the 'letter' value printed next to it, do **not** change anything to make it match: just send the notebook as it is.

In [1]:
!pip install -q hls4ml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 1.9 MB/s eta 0:00:00


In [2]:
# ============================================================
# SHARED SETUP — AUGMENTED data (4x Cheng et al. 2020), identical to the paper pipeline
# ============================================================
import os, sys, random, glob, json, time, zipfile, shutil, subprocess, warnings
import numpy as np
import pandas as pd
import scipy.signal, scipy.stats
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
def set_seeds(seed=SEED):
    random.seed(seed); np.random.seed(seed); tf.keras.utils.set_random_seed(seed)
set_seeds()

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

PROJECT_DIR = os.environ.get('PPG_PROJECT_DIR', '/content/drive/MyDrive/ppg_afib_project')
OUT_DIR = f'{PROJECT_DIR}/augmented_window_results'
os.makedirs(OUT_DIR, exist_ok=True)

FS = 125
WINDOW_SEC = 20
WINDOW_SAMPLES = FS * WINDOW_SEC   # 2,500 samples

# ---------- 1. FIND + unzip the two PERform CSV zips (searches Drive recursively, like your earlier notebook) ----------
import re
DATA = f'{PROJECT_DIR}/ppg_afib_project'      # folder where you uploaded the two zips (searched first)
AF_ZIP_MANUAL, NON_ZIP_MANUAL = None, None   # OPTIONAL: hard-code the two paths if auto-search picks the wrong files

def _find_zips(root):
    hits = []
    for dp, dns, fns in os.walk(root):
        dns[:] = [d for d in dns if d not in ('.Trash', '.Trash-0')]
        hits += [os.path.join(dp, f) for f in fns if f.lower().endswith('.zip')]
    return [z for z in hits if 'hls4ml' not in os.path.basename(z).lower()]
_is_non = lambda p: 'non' in os.path.basename(p).lower()
_is_af = lambda p: bool(re.search(r'(^|[^a-z])af([^a-z]|$)', os.path.basename(p).lower())) and not _is_non(p)

if not (os.path.isdir('af_data') and os.path.isdir('non_af_data')):
    if AF_ZIP_MANUAL and NON_ZIP_MANUAL:
        af_zip, non_zip = AF_ZIP_MANUAL, NON_ZIP_MANUAL
    else:
        zips = _find_zips(DATA) or _find_zips(os.path.dirname(PROJECT_DIR))
        print('candidate zips:', *zips, sep='\n  ')
        af_zip = next((z for z in zips if _is_af(z)), None); non_zip = next((z for z in zips if _is_non(z)), None)
    assert af_zip and non_zip, f'AF / non-AF zips not found - set AF_ZIP_MANUAL / NON_ZIP_MANUAL (looked under {DATA})'
    print('AF zip    :', af_zip); print('non-AF zip:', non_zip)
    for zp, d in ((af_zip, 'af_data'), (non_zip, 'non_af_data')):
        with zipfile.ZipFile(zp) as z:
            z.extractall(d)

af_files = sorted(glob.glob('af_data/**/mimic_perform_af_*_data.csv', recursive=True))
non_af_files = sorted(glob.glob('non_af_data/**/mimic_perform_non_af_*_data.csv', recursive=True))
print(f'Found {len(af_files)} AF files | {len(non_af_files)} non-AF files')
assert len(af_files) and len(non_af_files), 'CSV files not found after extraction - check the zip contents'

# ---------- 2. Original (un-augmented) 20 s windows, cached to Drive ----------
def load_windows(file_list, label):
    W, L, S = [], [], []
    for idx, fp in enumerate(file_list):
        ppg = pd.read_csv(fp)['PPG'].values.astype(np.float32)
        ppg = ppg[~np.isnan(ppg)]
        for i in range(len(ppg) // WINDOW_SAMPLES):
            W.append(ppg[i * WINDOW_SAMPLES:(i + 1) * WINDOW_SAMPLES])
            L.append(label); S.append(f'{label}_{idx}')   # same subject-ID scheme as the original notebook
    return W, L, S

CACHE = f'{PROJECT_DIR}/original_windows_cache.npz'
if os.path.exists(CACHE):
    _c = np.load(CACHE, allow_pickle=False)
    W_ALL, L_ALL, S_ALL = _c['W'], _c['L'], _c['S']
else:
    w1, l1, s1 = load_windows(af_files, 1)
    w0, l0, s0 = load_windows(non_af_files, 0)
    W_ALL = np.array(w1 + w0, dtype=np.float32)
    L_ALL = np.array(l1 + l0, dtype=np.int32)
    S_ALL = np.array(s1 + s0)
    np.savez_compressed(CACHE, W=W_ALL, L=L_ALL, S=S_ALL)
print(f'Original windows: {W_ALL.shape} | AF: {(L_ALL==1).sum()} | non-AF: {(L_ALL==0).sum()} | subjects: {len(np.unique(S_ALL))}')

# ---------- 3. Peak detection + the 11 HRV features (identical to original; parameters exposed) ----------
def extract_intervals(ppg, fs=FS, prom_frac=0.15, min_dist_s=0.4, plaus=(0.33, 1.5), plaus_frac=0.8):
    min_distance = int(min_dist_s * fs)
    rng = np.ptp(ppg)
    prom = prom_frac * rng if rng > 0 else 0
    peaks, _ = scipy.signal.find_peaks(ppg, distance=min_distance, prominence=prom)
    if len(peaks) < 2:
        return None
    iv = np.diff(peaks / fs)
    ok = (iv > plaus[0]) & (iv < plaus[1])
    if ok.mean() < plaus_frac:
        return None
    return iv

def compute_features(iv):
    if iv is None or len(iv) < 4:
        return None
    mean_iv, std_iv = float(np.mean(iv)), float(np.std(iv))
    rmssd = float(np.sqrt(np.mean(np.diff(iv) ** 2)))
    mn, mx = float(np.min(iv)), float(np.max(iv))
    cv = std_iv / mean_iv if mean_iv > 0 else 0.0
    d = np.abs(np.diff(iv))
    return [mean_iv, std_iv, rmssd, mn, mx, mx - mn, cv, float(len(iv) + 1),
            float(np.median(iv)), float(scipy.stats.skew(iv)), float(np.mean(d > 0.05))]

FEATURE_NAMES = ['mean_iv', 'std_iv', 'rmssd', 'min_iv', 'max_iv', 'range_iv', 'cv_iv', 'n_beats', 'median_iv', 'skew_iv', 'pnn50']

def build_features(W, L, S, **peak_kw):
    """Returns X, y, subj, kept_index (windows rejected by the quality gate are dropped, as in the original)."""
    X, y, s, kept = [], [], [], []
    for i in range(len(W)):
        f = compute_features(extract_intervals(W[i], **peak_kw))
        if f is not None:
            X.append(f); y.append(L[i]); s.append(S[i]); kept.append(i)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32), np.array(s), np.array(kept)

# ---------- 3b. Cheng et al. (2020) augmentation + quality gate, IDENTICAL to data_augmentation_pipeline.ipynb ----------
def apply_cheng_data_augmentation(segment, fs=FS):
    """original + Gaussian noise (5% of window SD) + amplitude scaling N(1, 0.15) + 0.2 Hz baseline wander (20% of window SD)"""
    out = [segment]
    out.append(segment + np.random.normal(0, 0.05 * np.std(segment), len(segment)))
    out.append(segment * np.random.normal(1.0, 0.15))
    t = np.linspace(0, len(segment) / fs, len(segment))
    out.append(segment + 0.2 * np.std(segment) * np.sin(2 * np.pi * 0.2 * t))
    return out

def build_features_augmented(W, L, S):
    X, y, s, dropped = [], [], [], {0: 0, 1: 0}
    for i in range(len(W)):                       # AF windows first, then non-AF (same order as the original notebook)
        for v in apply_cheng_data_augmentation(W[i], fs=FS):
            f = compute_features(extract_intervals(v))
            if f is not None:
                X.append(f); y.append(L[i]); s.append(S[i])
            else:
                dropped[int(L[i])] += 1
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32), np.array(s), dropped

np.random.seed(42); random.seed(42)               # same RNG state as CELL 1 of data_augmentation_pipeline.ipynb
X_ALL, y_ALL, subj_ALL, DROPPED = build_features_augmented(W_ALL, L_ALL, S_ALL)
print(f'AUGMENTED feature matrix: {X_ALL.shape} | AF: {(y_ALL==1).sum()} | non-AF: {(y_ALL==0).sum()} | dropped: {DROPPED}')
print('EXPECTED (paper): (7706, 11) | AF 4282 | non-AF 3424 | dropped AF 266, non-AF 404')
if X_ALL.shape[0] != 7706:
    print('WARNING: the dataset differs from the one in the paper. Check the CSV order / that the same two zips are used.')

# ---------- 4. Splits ----------
def holdout_subjects():
    """Same subject-level 80/20 stratified split (random_state=42) as CELL 6 of the original notebook."""
    uniq = np.unique(subj_ALL)
    labs = [y_ALL[subj_ALL == s][0] for s in uniq]
    tr, te = train_test_split(uniq, test_size=0.20, random_state=42, stratify=labs)
    return tr, te

def make_subject_folds(subj_array, label_array, n_folds=5, seed=SEED):
    """Stratified (by class) subject -> fold map. Fixed ONCE so every configuration uses identical test subjects."""
    rng = np.random.RandomState(seed)
    fmap = {}
    for cls in (0, 1):
        subs = np.unique(subj_array[label_array == cls]); rng.shuffle(subs)
        for k, s in enumerate(subs):
            fmap[s] = k % n_folds
    return fmap

FOLD_MAP = make_subject_folds(S_ALL, L_ALL, 5)

# ---------- 5. Model + evaluation helpers ----------
def make_mlp(n_in=11):
    return models.Sequential([layers.Input(shape=(n_in,)),
                              layers.Dense(16, activation='relu'), layers.Dropout(0.2),
                              layers.Dense(8, activation='relu'),
                              layers.Dense(1, activation='sigmoid')])

def fit_mlp(Xtr, ytr, epochs=30, seed=SEED):
    tf.keras.backend.clear_session(); set_seeds(seed)
    m = make_mlp(Xtr.shape[1])
    m.compile(optimizer=Adam(0.001), loss='binary_crossentropy', metrics=['accuracy'])
    m.fit(Xtr, ytr, epochs=epochs, batch_size=32, verbose=0)
    return m

def keras_fp(Xtr, ytr, Xte, epochs=30):
    sc = StandardScaler(); a = sc.fit_transform(Xtr); b = sc.transform(Xte)
    return fit_mlp(a, ytr, epochs).predict(b, verbose=0).flatten()

def sklearn_fp(factory):
    def fp(Xtr, ytr, Xte):
        sc = StandardScaler(); a = sc.fit_transform(Xtr); b = sc.transform(Xte)
        clf = factory(); clf.fit(a, ytr)
        if hasattr(clf, 'predict_proba'):
            return clf.predict_proba(b)[:, 1]
        return 1.0 / (1.0 + np.exp(-clf.decision_function(b)))
    return fp

def loso_predict(X, y, subj, fit_predict, verbose=True):
    """Leave-one-subject-out: returns out-of-subject probabilities aligned with X."""
    p = np.zeros(len(y), dtype=np.float64)
    uniq = np.unique(subj)
    for k, s in enumerate(uniq):
        te = subj == s
        p[te] = fit_predict(X[~te], y[~te], X[te])
        if verbose and (k + 1) % 5 == 0:
            print(f'  LOSO {k+1}/{len(uniq)}', end='\r')
    return p

def fold_predict(X, y, subj, fit_predict, fmap=None):
    """Fixed subject-level 5-fold CV; returns pooled out-of-fold probabilities + fold id per sample."""
    fmap = fmap or FOLD_MAP
    fold = np.array([fmap[s] for s in subj]); p = np.zeros(len(y))
    for k in range(max(fmap.values()) + 1):
        te = fold == k
        p[te] = fit_predict(X[~te], y[~te], X[te])
    return p, fold

# ---------- 6. Subject-clustered statistics (windows from one subject are NOT independent) ----------
def subject_counts(y, p, subj, thr=0.5):
    pred = (p >= thr).astype(int); rows = []
    for s in np.unique(subj):
        m = subj == s; yy, pp = y[m], pred[m]
        assert len(np.unique(yy)) == 1, 'a subject has mixed labels'
        rows.append((s, int(yy[0]), int(((pp == 1) & (yy == 1)).sum()), int(((pp == 1) & (yy == 0)).sum()),
                     int(((pp == 0) & (yy == 0)).sum()), int(((pp == 0) & (yy == 1)).sum())))
    return pd.DataFrame(rows, columns=['subject', 'label', 'tp', 'fp', 'tn', 'fn'])

def _m(tp, fp, tn, fn):
    n = tp + fp + tn + fn
    sens = tp / (tp + fn) if tp + fn else np.nan
    spec = tn / (tn + fp) if tn + fp else np.nan
    prec = tp / (tp + fp) if tp + fp else np.nan
    f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else np.nan
    return {'accuracy': (tp + tn) / n, 'sensitivity': sens, 'specificity': spec, 'precision': prec, 'f1': f1}

def cluster_bootstrap(y, p, subj, n_boot=5000, seed=SEED, thr=0.5, do_auc=True, n_boot_auc=1000):
    """Class-stratified subject bootstrap (resample AF subjects among AF, non-AF among non-AF).
    Returns {metric: (point, lo95, hi95)}; also 'per_subject_acc_mean' (macro average over subjects)."""
    rng = np.random.RandomState(seed)
    C = subject_counts(y, p, subj, thr)
    cnt = C[['tp', 'fp', 'tn', 'fn']].values.astype(float)
    af = np.where(C['label'].values == 1)[0]; nf = np.where(C['label'].values == 0)[0]
    point = _m(*cnt.sum(0))
    persub = ((C.tp + C.tn) / (C.tp + C.fp + C.tn + C.fn)).values
    point['per_subject_acc_mean'] = persub.mean()
    draws = {k: [] for k in point}
    for _ in range(n_boot):
        idx = np.concatenate([rng.choice(af, len(af)), rng.choice(nf, len(nf))])
        r = _m(*cnt[idx].sum(0)); r['per_subject_acc_mean'] = persub[idx].mean()
        for k in r: draws[k].append(r[k])
    out = {k: (point[k], *np.nanpercentile(draws[k], [2.5, 97.5])) for k in point}
    if do_auc:
        subs = C['subject'].values
        by = {s: np.where(subj == s)[0] for s in subs}
        out['auc'] = (roc_auc_score(y, p), np.nan, np.nan)
        a = []
        for _ in range(n_boot_auc):
            idx = np.concatenate([rng.choice(af, len(af)), rng.choice(nf, len(nf))])
            ii = np.concatenate([by[subs[j]] for j in idx])
            if len(np.unique(y[ii])) == 2:
                a.append(roc_auc_score(y[ii], p[ii]))
        out['auc'] = (out['auc'][0], *np.percentile(a, [2.5, 97.5]))
    return out

def paired_bootstrap_diff(y, pA, pB, subj, n_boot=5000, seed=SEED, thr=0.5):
    """Accuracy(A) - Accuracy(B) with a shared subject resample. Returns (diff, lo95, hi95)."""
    rng = np.random.RandomState(seed)
    CA, CB = subject_counts(y, pA, subj, thr), subject_counts(y, pB, subj, thr)
    a = CA[['tp', 'fp', 'tn', 'fn']].values.astype(float); b = CB[['tp', 'fp', 'tn', 'fn']].values.astype(float)
    af = np.where(CA['label'].values == 1)[0]; nf = np.where(CA['label'].values == 0)[0]
    acc = lambda c: (c[0] + c[2]) / c.sum()
    d = []
    for _ in range(n_boot):
        idx = np.concatenate([rng.choice(af, len(af)), rng.choice(nf, len(nf))])
        d.append(acc(a[idx].sum(0)) - acc(b[idx].sum(0)))
    return acc(a.sum(0)) - acc(b.sum(0)), *np.percentile(d, [2.5, 97.5])

def wilson_ci(k, n, z=1.96):
    """NAIVE window-level Wilson interval — anti-conservative here (windows are clustered by subject)."""
    ph = k / n; den = 1 + z ** 2 / n
    c = (ph + z ** 2 / (2 * n)) / den
    h = z * np.sqrt(ph * (1 - ph) / n + z ** 2 / (4 * n ** 2)) / den
    return c - h, c + h

def fmt(t, pct=True):
    f = 100 if pct else 1
    return f'{t[0]*f:.2f} [{t[1]*f:.2f}, {t[2]*f:.2f}]'

print('Setup complete.')

Mounted at /content/drive
candidate zips:
  /content/drive/MyDrive/Classroom/C Programming  (Fall 2025) Section 4/s20250020287_sriramchandar reddy (1).zip
  /content/drive/MyDrive/Classroom/C Programming  (Fall 2025) Section 4/s20250020287_assignment2 sriramchandarreddy.zip
  /content/drive/MyDrive/Classroom/C Programming  (Fall 2025) Section 4/S20250020287_ASSIGNMENT2 SRIRAM.zip
  /content/drive/MyDrive/Classroom/C Programming  (Fall 2025) Section 4/s20250020287_assignment-3 sriram.zip
  /content/drive/MyDrive/Classroom/C Programming  (Fall 2025) Section 4/s20250020287 cp lab assignment-4.zip
  /content/drive/MyDrive/Classroom/C Programming  (Fall 2025) Section 4/s20250020287_assignment-5.zip
  /content/drive/MyDrive/Classroom/C Programming  (Fall 2025) Section 4/s20250020287_assignment-6.zip
  /content/drive/MyDrive/Classroom/C Programming  (Fall 2025) Section 4/s20250020287_sriramchandar reddy.zip
  /content/drive/MyDrive/Classroom/C Programming  (Fall 2025) Section 4/s20250020287_a

In [3]:
def find_h5():
    for root in (PROJECT_DIR, '/content/drive/MyDrive'):
        hits = []
        for dp, dns, fns in os.walk(root):
            dns[:] = [d for d in dns if not d.startswith('.Trash')]
            if 'ppg_afib_model_augmented.h5' in fns: hits.append(os.path.join(dp, 'ppg_afib_model_augmented.h5'))
        if hits: return sorted(hits, key=lambda s: (len(s), s))[0]
    return None

def split_scaled(X, y, subj):
    tr_s, te_s = holdout_subjects()
    tr, te = np.isin(subj, tr_s), np.isin(subj, te_s)
    sc = StandardScaler().fit(X[tr])
    return sc.transform(X[tr]).astype(np.float32), y[tr], sc.transform(X[te]).astype(np.float32), y[te]

# ---------- hls4ml bit-accurate simulation helpers (same recipe as notebooks 07 / 09) ----------
import hls4ml, tempfile
def fx(w, i, rnd='RND', ovf='SAT'): return f'ap_fixed<{w},{i},AP_{rnd},AP_{ovf}>'
U_INT = {16: 6, 12: 5, 6: 4, 4: 3}   # widths that use ONE common fixed<W,I> for every tensor (paper: 16 -> <16,6>, 12 -> <12,5>, 6 -> <6,4>; the 4-bit integer part is an assumption)
def dense_names(model): return [l.name for l in model.layers if isinstance(l, layers.Dense)]

def make_cfg(model, width=8, rnd='RND', ovf='SAT', in_int=5, accum=(16, 8), out=(12, 2), reuse=1, strategy='Latency'):
    """width 8 = the paper's profiled design: input and layer outputs <8,5>, hidden weights/biases <8,1>, output-layer weights/biases <8,2>,
    accumulators <16,8>, output <12,2>. Widths in U_INT use one common fixed<W,I> for every tensor."""
    cfg = hls4ml.utils.config_from_keras_model(model, granularity='name')
    cfg['Model']['ReuseFactor'] = reuse; cfg['Model']['Strategy'] = strategy
    if width in U_INT:
        f = fx(width, U_INT[width], rnd, ovf)
        cfg['Model']['Precision']['default'] = f
        for ln in cfg['LayerName']:
            p = cfg['LayerName'][ln].get('Precision', None)
            if isinstance(p, dict):
                for k in p: p[k] = f
        return cfg
    assert width == 8, 'width must be 8 (profiled) or one of ' + str(list(U_INT))
    names = dense_names(model)
    cfg['Model']['Precision']['default'] = fx(8, 5, rnd, ovf)
    cfg['Model']['Precision']['result'] = fx(*out, rnd, ovf)
    for ln in cfg['LayerName']:
        if 'input' in ln.lower():
            cfg['LayerName'][ln].setdefault('Precision', {})
            if isinstance(cfg['LayerName'][ln]['Precision'], dict): cfg['LayerName'][ln]['Precision']['result'] = fx(8, in_int, rnd, ovf)
    for n in names[:-1]:
        cfg['LayerName'][n]['Precision'].update(weight=fx(8, 1, rnd, ovf), bias=fx(8, 1, rnd, ovf), result=fx(8, 5, rnd, ovf), accum=fx(*accum, rnd, ovf))
        cfg['LayerName'][n]['ReuseFactor'] = reuse
    cfg['LayerName'][names[-1]]['Precision'].update(weight=fx(8, 2, rnd, ovf), bias=fx(8, 2, rnd, ovf), result=fx(*out, rnd, ovf), accum=fx(*accum, rnd, ovf))
    cfg['LayerName'][names[-1]]['ReuseFactor'] = reuse
    return cfg

_SIM_N = [0]
def bit_accurate_predict(model, cfg, X, keep=False):
    """Compile the hls4ml C simulation of this configuration and return the output probabilities for X."""
    _SIM_N[0] += 1
    d = os.path.join(tempfile.gettempdir(), f'hls_sim_{os.getpid()}_{_SIM_N[0]}')
    shutil.rmtree(d, ignore_errors=True)
    h = hls4ml.converters.convert_from_keras_model(model, hls_config=cfg, output_dir=d, backend='Vivado')
    h.compile()
    p = h.predict(np.ascontiguousarray(X, dtype=np.float32)).flatten()
    if not keep: shutil.rmtree(d, ignore_errors=True)
    return p

def acc_of(p, y): return float(np.mean((np.asarray(p) > 0.5) == (np.asarray(y) > 0.5)))
def ptq_acc(model, X, y, **kw): return acc_of(bit_accurate_predict(model, make_cfg(model, **kw), X), y)
print('hls4ml', hls4ml.__version__)

# ---------- quantization-aware fine-tuning (fake quantization: round-to-nearest + saturation, straight-through gradient) ----------
def fq(x, w, i):
    sc = 2.0 ** (w - i)
    r = tf.clip_by_value(tf.floor(x * sc + 0.5), -2.0 ** (w - 1), 2.0 ** (w - 1) - 1) / sc
    return x + tf.stop_gradient(r - x)

QFMT = {8: dict(a=(8, 5), wh=(8, 1), wo=(8, 2), out=(12, 2)),
        6: dict(a=(6, 4), wh=(6, 4), wo=(6, 4), out=(6, 4)),
        4: dict(a=(4, 3), wh=(4, 3), wo=(4, 3), out=(4, 3))}      # same formats as make_cfg()

class QDense(layers.Layer):
    def __init__(self, units, relu, wfmt, afmt, qin=None, **kw):
        super().__init__(**kw); self.units, self.relu, self.wfmt, self.afmt, self.qin = units, relu, wfmt, afmt, qin
    def build(self, input_shape):
        self.k = self.add_weight(shape=(int(input_shape[-1]), self.units), initializer='glorot_uniform', name='k')
        self.b = self.add_weight(shape=(self.units,), initializer='zeros', name='b')
    def call(self, x):
        if self.qin is not None: x = fq(x, *self.qin)
        y = tf.matmul(x, fq(self.k, *self.wfmt)) + fq(self.b, *self.wfmt)
        if self.relu: y = tf.nn.relu(y)
        return fq(y, *self.afmt)

def make_qat_model(width):
    f = QFMT[width]
    inp = layers.Input(shape=(11,))
    x = QDense(16, True, f['wh'], f['a'], qin=f['a'])(inp); x = layers.Dropout(0.2)(x)
    x = QDense(8, True, f['wh'], f['a'])(x)
    z = QDense(1, False, f['wo'], f['out'])(x)
    return models.Model(inp, layers.Activation('sigmoid')(z))

def qat_finetune(float_model, Xtr, ytr, width, epochs=30, lr=0.0005, seed=SEED):
    """Fine-tune a copy of the trained float network for 30 epochs, lr 5e-4, with the fixed-point arithmetic in the forward pass.
    Returns (plain_keras_model_with_the_tuned_weights, fake_quant_model)."""
    set_seeds(seed)
    q = make_qat_model(width)
    dn = [l for l in float_model.layers if isinstance(l, layers.Dense)]
    ql = [l for l in q.layers if isinstance(l, QDense)]
    for a, b in zip(dn, ql): b.set_weights(a.get_weights())
    q.compile(optimizer=Adam(lr), loss='binary_crossentropy')
    q.fit(Xtr, ytr, epochs=epochs, batch_size=32, verbose=0)
    plain = make_mlp(Xtr.shape[1])
    for a, b in zip([l for l in plain.layers if isinstance(l, layers.Dense)], ql): a.set_weights(b.get_weights())
    return plain, q

def compare_ptq_qat(float_model, Xtr, ytr, Xte, yte, widths=(8, 6, 4), tag=''):
    """One row per width: float / PTQ / QAT accuracy (all evaluated with the hls4ml bit-accurate simulation, RND + SAT)."""
    pf = float_model.predict(Xte, verbose=0).flatten(); rows = []
    for w in widths:
        a_ptq = ptq_acc(float_model, Xte, yte, width=w)
        plain, q = qat_finetune(float_model, Xtr, ytr, w)
        p_sim = bit_accurate_predict(plain, make_cfg(plain, width=w), Xte)
        agree = float(np.mean((q.predict(Xte, verbose=0).flatten() > 0.5) == (p_sim > 0.5)))
        rows.append(dict(set=tag, bits=w, float_acc=acc_of(pf, yte), ptq_acc=a_ptq, qat_acc=acc_of(p_sim, yte), fakequant_vs_sim_agreement=agree))
        print(rows[-1])
    return rows

hls4ml 1.3.0


## A. Original windows - 348-window held-out test set (7 held-out subjects)

In [4]:
Xo, yo, so, _ = build_features(W_ALL, L_ALL, S_ALL)
Xtr, ytr, Xte, yte = split_scaled(Xo, yo, so)
mo = fit_mlp(Xtr, ytr)
print('train', len(Xtr), 'test', len(Xte), '| float test accuracy %.2f%%  (letter: 97.99%%)' % (100 * acc_of(mo.predict(Xte, verbose=0).flatten(), yte)))
rows = compare_ptq_qat(mo, Xtr, ytr, Xte, yte, tag='original, 348 test windows')

train 1572 test 348 | float test accuracy 97.99%  (letter: 97.99%)
{'set': 'original, 348 test windows', 'bits': 8, 'float_acc': 0.9798850574712644, 'ptq_acc': 0.9798850574712644, 'qat_acc': 0.9827586206896551, 'fakequant_vs_sim_agreement': 1.0}
{'set': 'original, 348 test windows', 'bits': 6, 'float_acc': 0.9798850574712644, 'ptq_acc': 0.9741379310344828, 'qat_acc': 0.9770114942528736, 'fakequant_vs_sim_agreement': 0.9971264367816092}
{'set': 'original, 348 test windows', 'bits': 4, 'float_acc': 0.9798850574712644, 'ptq_acc': 0.9597701149425287, 'qat_acc': 0.9741379310344828, 'fakequant_vs_sim_agreement': 0.9913793103448276}


## B. Original windows - 5-fold cross-validation over subjects (fixed `FOLD_MAP`)

In [5]:
def cv_ptq_qat(X, y, subj, tag, widths=(8, 6, 4)):
    fold = np.array([FOLD_MAP[s] for s in subj]); res = []
    for k in range(5):
        te = fold == k
        if te.sum() == 0: continue
        sc = StandardScaler().fit(X[~te]); a = sc.transform(X[~te]).astype(np.float32); b = sc.transform(X[te]).astype(np.float32)
        m = fit_mlp(a, y[~te])
        r = compare_ptq_qat(m, a, y[~te], b, y[te], widths, tag=f'{tag} fold {k+1}')
        res += [dict(r_, fold=k + 1, n_test=int(te.sum())) for r_ in r]
    return pd.DataFrame(res)
cvA = cv_ptq_qat(Xo, yo, so, 'original CV')
print(cvA.groupby('bits')[['float_acc', 'ptq_acc', 'qat_acc']].agg(['mean', 'std']).mul(100).round(2))

{'set': 'original CV fold 1', 'bits': 8, 'float_acc': 0.9635193133047211, 'ptq_acc': 0.9656652360515021, 'qat_acc': 0.9463519313304721, 'fakequant_vs_sim_agreement': 0.9978540772532188}
{'set': 'original CV fold 1', 'bits': 6, 'float_acc': 0.9635193133047211, 'ptq_acc': 0.9549356223175965, 'qat_acc': 0.9613733905579399, 'fakequant_vs_sim_agreement': 0.9914163090128756}
{'set': 'original CV fold 1', 'bits': 4, 'float_acc': 0.9635193133047211, 'ptq_acc': 0.9399141630901288, 'qat_acc': 0.9527896995708155, 'fakequant_vs_sim_agreement': 0.9828326180257511}
{'set': 'original CV fold 2', 'bits': 8, 'float_acc': 0.9776674937965261, 'ptq_acc': 0.9702233250620348, 'qat_acc': 0.9602977667493796, 'fakequant_vs_sim_agreement': 0.9975186104218362}
{'set': 'original CV fold 2', 'bits': 6, 'float_acc': 0.9776674937965261, 'ptq_acc': 0.9875930521091811, 'qat_acc': 0.9776674937965261, 'fakequant_vs_sim_agreement': 0.967741935483871}
{'set': 'original CV fold 2', 'bits': 4, 'float_acc': 0.977667493796526

## C. Augmented windows - 5-fold cross-validation over subjects

In [6]:
cvB = cv_ptq_qat(X_ALL, y_ALL, subj_ALL, 'augmented CV')
print(cvB.groupby('bits')[['float_acc', 'ptq_acc', 'qat_acc']].agg(['mean', 'std']).mul(100).round(2))

{'set': 'augmented CV fold 1', 'bits': 8, 'float_acc': 0.9678456591639871, 'ptq_acc': 0.9630225080385852, 'qat_acc': 0.9581993569131833, 'fakequant_vs_sim_agreement': 0.9989281886387996}
{'set': 'augmented CV fold 1', 'bits': 6, 'float_acc': 0.9678456591639871, 'ptq_acc': 0.9801714898177921, 'qat_acc': 0.9769560557341908, 'fakequant_vs_sim_agreement': 0.9903536977491961}
{'set': 'augmented CV fold 1', 'bits': 4, 'float_acc': 0.9678456591639871, 'ptq_acc': 0.9790996784565916, 'qat_acc': 0.9437299035369775, 'fakequant_vs_sim_agreement': 0.9523043944265809}
{'set': 'augmented CV fold 2', 'bits': 8, 'float_acc': 0.9615145872129113, 'ptq_acc': 0.9739292364990689, 'qat_acc': 0.9503414028553694, 'fakequant_vs_sim_agreement': 1.0}
{'set': 'augmented CV fold 2', 'bits': 6, 'float_acc': 0.9615145872129113, 'ptq_acc': 0.9745499689633768, 'qat_acc': 0.9540657976412167, 'fakequant_vs_sim_agreement': 0.9813780260707635}
{'set': 'augmented CV fold 2', 'bits': 4, 'float_acc': 0.9615145872129113, 'ptq_

## D. Augmented windows - 1,394-window test set

In [7]:
Xtr2, ytr2, Xte2, yte2 = split_scaled(X_ALL, y_ALL, subj_ALL)
p = find_h5(); print('paper model:', p)
mp = tf.keras.models.load_model(p, compile=False)
print('paper network, float: %.2f%%  (letter 98.06%%)' % (100 * acc_of(mp.predict(Xte2, verbose=0).flatten(), yte2)))
rowsD = []
for w in (16, 12, 8, 6):
    rowsD.append(dict(bits=w, ptq_acc=ptq_acc(mp, Xte2, yte2, width=w))); print(rowsD[-1])
print('letter: 16 bit 98.06 | 12 bit 98.13 | 8 bit 98.35 | 6 bit 97.63')
# a freshly trained float network (shows that PTQ varies by about one point between trained networks)
mr = fit_mlp(Xtr2, ytr2)
print('re-trained float network: %.2f%%  (letter 98.28%%)' % (100 * acc_of(mr.predict(Xte2, verbose=0).flatten(), yte2)))
rowsE = compare_ptq_qat(mr, Xtr2, ytr2, Xte2, yte2, widths=(8, 6), tag='augmented, re-trained network, 1394 test windows')

paper model: /content/drive/MyDrive/ppg_afib_project/ppg_afib_model_augmented.h5
paper network, float: 98.06%  (letter 98.06%)
{'bits': 16, 'ptq_acc': 0.9806312769010043}
{'bits': 12, 'ptq_acc': 0.9813486370157819}
{'bits': 8, 'ptq_acc': 0.9835007173601148}
{'bits': 6, 'ptq_acc': 0.9763271162123386}
letter: 16 bit 98.06 | 12 bit 98.13 | 8 bit 98.35 | 6 bit 97.63
re-trained float network: 98.28%  (letter 98.28%)
{'set': 'augmented, re-trained network, 1394 test windows', 'bits': 8, 'float_acc': 0.9827833572453372, 'ptq_acc': 0.9748923959827833, 'qat_acc': 0.9619799139167863, 'fakequant_vs_sim_agreement': 0.9992826398852224}
{'set': 'augmented, re-trained network, 1394 test windows', 'bits': 6, 'float_acc': 0.9827833572453372, 'ptq_acc': 0.9784791965566715, 'qat_acc': 0.975609756097561, 'fakequant_vs_sim_agreement': 0.9878048780487805}


## Save everything

In [8]:
allrows = pd.concat([pd.DataFrame(rows), cvA, cvB, pd.DataFrame(rowsD), pd.DataFrame(rowsE)], ignore_index=True)
allrows.to_csv(f'{OUT_DIR}/qat_vs_ptq_results.csv', index=False)
print('saved', f'{OUT_DIR}/qat_vs_ptq_results.csv'); display(allrows.round(4))

saved /content/drive/MyDrive/ppg_afib_project/augmented_window_results/qat_vs_ptq_results.csv


,set,bits,float_acc,ptq_acc,qat_acc,fakequant_vs_sim_agreement,fold,n_test
0,"original, 348 test windows",8,0.9799,0.9799,0.9828,1.0000,NaN,NaN
1,"original, 348 test windows",6,0.9799,0.9741,0.9770,0.9971,NaN,NaN
2,"original, 348 test windows",4,0.9799,0.9598,0.9741,0.9914,NaN,NaN
3,original CV fold 1,8,0.9635,0.9657,0.9464,0.9979,1.0,466.0
4,original CV fold 1,6,0.9635,0.9549,0.9614,0.9914,1.0,466.0
5,original CV fold 1,4,0.9635,0.9399,0.9528,0.9828,1.0,466.0
6,original CV fold 2,8,0.9777,0.9702,0.9603,0.9975,2.0,403.0
7,original CV fold 2,6,0.9777,0.9876,0.9777,0.9677,2.0,403.0
8,original CV fold 2,4,0.9777,0.9330,0.9355,0.9851,2.0,403.0
9,original CV fold 3,8,0.9293,0.9266,0.8098,0.9973,3.0,368.0


In [ ]:
# ===== NEW CELL E: QAT on the PAPER network, 1,394 augmented test windows =====
Xtr2, ytr2, Xte2, yte2 = split_scaled(X_ALL, y_ALL, subj_ALL)
mp = tf.keras.models.load_model(find_h5(), compile=False)
print('paper network, float: %.2f%%  (letter 98.06%%)' % (100 * acc_of(mp.predict(Xte2, verbose=0).flatten(), yte2)))
rowsF = compare_ptq_qat(mp, Xtr2, ytr2, Xte2, yte2, widths=(8, 6, 4), tag='augmented, paper network, 1394 test windows')
print('letter PTQ values to reproduce: 8 bit 98.35 | 6 bit 97.63')

In [ ]:
# ===== NEW CELL F: save =====
try:
    allrows2 = pd.concat([pd.DataFrame(rows), cvA, cvB, pd.DataFrame(rowsF)], ignore_index=True)
    allrows2.to_csv(f'{OUT_DIR}/qat_vs_ptq_results_paper_network.csv', index=False)
    print(f'Successfully saved combined results to {OUT_DIR}/qat_vs_ptq_results_paper_network.csv')
except NameError:
    pd.DataFrame(rowsF).to_csv(f'{OUT_DIR}/qat_vs_ptq_results_paper_network.csv', index=False)
    print(f'Saved rowsF results to {OUT_DIR}/qat_vs_ptq_results_paper_network.csv')

print('\n=== SUMMARY TABLE (rowsF) ===')
print(pd.DataFrame(rowsF)[['bits', 'float_acc', 'ptq_acc', 'qat_acc', 'fakequant_vs_sim_agreement']].round(4))